# Text Mining — Final Solution
**Spring Semester 2025/2026 | NOVA IMS**

**Model:** RoBERTa fine-tuned end-to-end (`roberta-base`)  
**Task:** 3-class tweet sentiment — Bearish (0) · Bullish (1) · Neutral (2)  
**Pipeline:** Minimal text cleaning → Optuna-tuned HuggingFace Trainer → predict on test.csv → pred_xx.csv

> Run all cells top-to-bottom. No external variables needed.  
> GPU strongly recommended (CUDA). CPU works but is slower (~2h for training).

## 1. Installs & Imports

In [1]:
!pip install transformers torch datasets accelerate optuna -q

In [2]:
import re, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from sklearn.metrics import classification_report, accuracy_score, f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
print(f'Device: {DEVICE}')
print(f'PyTorch {torch.__version__} | Transformers ready')

Device: cuda
PyTorch 2.8.0+cu128 | Transformers ready


## 2. Load Data

In [3]:
df_train = pd.read_csv('train.csv')
df_test  = pd.read_csv('test.csv')

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')
print(f'\nClass distribution (train):')
print(df_train['label'].value_counts().sort_index()
      .rename({0:'Bearish',1:'Bullish',2:'Neutral'}))

Train: (9543, 2)  |  Test: (2388, 2)

Class distribution (train):
label
Bearish    1442
Bullish    1923
Neutral    6178
Name: count, dtype: int64


## 3. Text Preprocessing

Minimal cleaning for transformer models: remove URLs and normalise whitespace.  
The RoBERTa tokeniser handles subword splitting, punctuation, and casing internally.  
Aggressive cleaning (lowercasing, stopword removal, lemmatisation) would destroy
information the model was pre-trained to use.

In [4]:
def preprocess_transformer(text_list):
    """
    Minimal cleaning for RoBERTa:
      - Remove URLs (no semantic value)
      - Collapse extra whitespace
    Everything else (casing, punctuation, @mentions, $tickers) is preserved.
    """
    results = []
    for text in tqdm(text_list, desc='Preprocessing'):
        text = str(text)
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        results.append(text)
    return results

print('Preprocessing train...')
X_train_raw = preprocess_transformer(df_train['text'].tolist())
y_train     = df_train['label'].tolist()

print('Preprocessing test...')
X_test_raw  = preprocess_transformer(df_test['text'].tolist())

print(f'\nExample:')
print(f'  Original : {df_train["text"].iloc[0]}')
print(f'  Cleaned  : {X_train_raw[0]}')

Preprocessing train...


Preprocessing: 100%|██████████| 9543/9543 [00:00<00:00, 135843.81it/s]


Preprocessing test...


Preprocessing: 100%|██████████| 2388/2388 [00:00<00:00, 128611.39it/s]


Example:
  Original : $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
  Cleaned  : $BYND - JPMorgan reels in expectations on Beyond Meat


## 4. Class Weights

The dataset is imbalanced (Neutral ~65%, Bullish ~20%, Bearish ~15%).  
Class weights penalise errors on minority classes proportionally.

In [5]:
label_counts = pd.Series(y_train).value_counts().sort_index()
total        = len(y_train)
class_weights = torch.tensor(
    [total / (3 * label_counts[i]) for i in range(3)],
    dtype=torch.float
)
print(f'Class weights:')
print(f'  Bearish (0): {class_weights[0]:.3f}')
print(f'  Bullish (1): {class_weights[1]:.3f}')
print(f'  Neutral (2): {class_weights[2]:.3f}')

Class weights:
  Bearish (0): 2.206
  Bullish (1): 1.654
  Neutral (2): 0.515


## 5. Tokenisation

RoBERTa uses a BPE tokeniser.  
`max_length=128` covers >99% of tweets without truncation.

In [6]:
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 128

print(f'Loading tokeniser: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
    )

# Full training set — no validation split (model already selected in tm_tests_xx)
train_hf = Dataset.from_dict({'text': X_train_raw, 'label': y_train})
test_hf  = Dataset.from_dict({'text': X_test_raw})

print('Tokenising train...')
train_hf = train_hf.map(tokenize_fn, batched=True)
print('Tokenising test...')
test_hf  = test_hf.map(tokenize_fn, batched=True)

print(f'\nTrain size : {len(train_hf)}')
print(f'Test size  : {len(test_hf)}')

Loading tokeniser: roberta-base


Tokenising train...


Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

Tokenising test...


Map:   0%|          | 0/2388 [00:00<?, ? examples/s]


Train size : 9543
Test size  : 2388


## 6. Model — RoBERTa Fine-tuned

**Why RoBERTa?**  
RoBERTa (`roberta-base`) is an improved version of BERT trained with more data,
larger batches, and without Next Sentence Prediction. It achieved the best
F1-Macro score across all models evaluated in `tm_tests_xx`, outperforming
domain-specific models (FinBERT, FinTwitBERT) on this dataset.

**Hyperparameters** selected via Optuna (10 trials on 2000-tweet subsample) in `tm_tests_xx`.

In [7]:
# Best hyperparameters from Optuna
# Replace these values with the actual best params from your Optuna study
BEST_LR         = 2.9045790726652738e-05    # best['learning_rate']
BEST_WD         = 0.038053996848046986    # best['weight_decay']
BEST_WARMUP     = 0.10401360423556216     # best['warmup_ratio']
BEST_BATCH      = 8      # best['batch_size']

print('Hyperparameters (from Optuna in tm_tests_xx):')
print(f'  learning_rate : {BEST_LR}')
print(f'  weight_decay  : {BEST_WD}')
print(f'  warmup_ratio  : {BEST_WARMUP}')
print(f'  batch_size    : {BEST_BATCH}')

Hyperparameters (from Optuna in tm_tests_xx):
  learning_rate : 2.9045790726652738e-05
  weight_decay  : 0.038053996848046986
  warmup_ratio  : 0.10401360423556216
  batch_size    : 8


In [8]:
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: 'Bearish', 1: 'Bullish', 2: 'Neutral'},
    label2id={'Bearish': 0, 'Bullish': 1, 'Neutral': 2},
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

Loading model: roberta-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 124,647,939


## 7. Fine-tuning

Training on the **full training set** (all tweets, no validation split).  
Class weights are applied via a custom Trainer to handle class imbalance.

In [9]:
class WeightedTrainer(Trainer):
    """Custom Trainer that applies class weights to the cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get('labels')
        outputs = model(**inputs)
        logits  = outputs.get('logits')
        loss    = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )(logits, labels)
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir                  = './roberta_final',
    num_train_epochs            = 5,
    learning_rate               = BEST_LR,
    per_device_train_batch_size = BEST_BATCH,
    per_device_eval_batch_size  = 32,
    warmup_ratio                = BEST_WARMUP,
    weight_decay                = BEST_WD,
    logging_steps               = 50,
    save_strategy               = 'no',
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer = WeightedTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_hf,
)

print('Starting fine-tuning on full training set...')
print(f'  Epochs  : {training_args.num_train_epochs}')
print(f'  Batch   : {BEST_BATCH}')
print(f'  Samples : {len(train_hf)}')
print(f'  fp16    : {training_args.fp16}')
print()
trainer.train()
print('\nFine-tuning complete.')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning on full training set...
  Epochs  : 5
  Batch   : 8
  Samples : 9543
  fp16    : True



Step,Training Loss
50,1.106294
100,1.102396
150,1.096352
200,0.993050
250,0.720078
300,0.615165
350,0.673784
400,0.587334
450,0.520882
500,0.751936



Fine-tuning complete.


## 8. Predict on test.csv

In [10]:
print('Generating predictions on test set...')
preds_output = trainer.predict(test_hf)
final_preds  = preds_output.predictions.argmax(axis=1)

label_map   = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
pred_series = pd.Series(final_preds)
print(f'\nPrediction distribution:')
for label, cnt in pred_series.value_counts().sort_index().items():
    print(f'  {label_map[label]} ({label}): {cnt} ({cnt/len(final_preds)*100:.1f}%)')

Generating predictions on test set...



Prediction distribution:
  Bearish (0): 364 (15.2%)
  Bullish (1): 513 (21.5%)
  Neutral (2): 1511 (63.3%)


## 9. Save Submission File

In [12]:
submission = pd.DataFrame({
    'id'   : df_test['id'],
    'label': final_preds,
})

submission.to_csv('pred_40.csv', index=False)
print('Saved: pred_40.csv')
print(f'Rows: {len(submission)}')
print(submission.head(10).to_string(index=False))

Saved: pred_40.csv
Rows: 2388
 id  label
  0      1
  1      2
  2      2
  3      1
  4      2
  5      1
  6      0
  7      0
  8      2
  9      2
